# Barranquilla MOPAC: una reacción enzimática en la GPU

Primer paso de la acilación en una serina proteasa: el oxígeno OG de **Ser195** de la tripsina ataca el carbono
carbonílico de **Lys15** de BPTI (PDB 2PTC; el enlace Lys15–Ala16 ocupa el sitio activo como sustrato) mientras
**His57** toma el protón de Ser195. `scripts/qmmm/qmmm_scan.py` hace un barrido QM/MM (mapeo adiabático) sobre dos
coordenadas a la vez: la distancia Ser195 OG – Lys15 C y la transferencia del protón, d(OG–HG) − d(HG–NE2 de His57)
(−1,1 Å: el protón en Ser195; +1 Å: en His57). Prepara el sistema como `run_qmmm_md.py` (hidrógenos, amber14 + TIP3P,
caja de agua, equilibrado MM y minimización MM de todo el sistema), minimiza el reactivo con QM/MM y recorre una malla
de las dos coordenadas con restricciones armónicas, fila por fila; en cada punto minimiza el sitio activo (los residuos
a menos de 10 Å de la región QM; el resto queda sujeto donde lo dejó la minimización MM) partiendo del punto anterior.
Al final vuelve al reactivo por el borde de la malla (última fila y primera columna), y la diferencia entre ida y
vuelta en los mismos puntos mide cuánto depende el mapa del camino. Por último (`RELEASE`) quita las restricciones en
cada punto de la última fila y minimiza: si la estructura se queda donde la dejó el barrido (por ejemplo el aducto
tetraédrico, con el protón en His57), está en un mínimo local de la energía; si se va, no. Con una sola coordenada
(OG–C) y el protón libre, un
barrido en CPU dio una subida monótona de la energía sin que el protón pasara a His57: hacen falta las dos.
Región QM: cadenas laterales de His57 (HID), Asp102 y Ser195 y los residuos Lys15–Ala16 de BPTI (54 átomos + 5 de
enlace), PM7/MOZYME en la GPU; la electrostática QM–MM de largo alcance por PME.

| Bloque | Qué hace | Tiempo (aprox.) |
|---|---|---|
| Compilar | MOPAC con la GPU | ~10 min |
| Mapas | una malla de 168 puntos, 13 de vuelta y 12 minimizaciones sin restricciones por semilla (equilibrado MM propio) | ~10 min cada una en una L4 sin `RELEASE` (medido, commit e799de85); `RELEASE` añade unos 6 min (estimado) |
| Figuras | el mapa de energía de cada réplica | segundos |

**Qué es y qué no es:** cada punto es una estructura minimizada; el mapa es de energía potencial, no de energía
libre (sin muestreo, entropía ni energía de punto cero), y depende del orden en que se recorre la malla y de la
estructura de partida: por eso hay tres réplicas y la vuelta.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU* (una L4 basta; A100 o T4 también sirven), y luego *Ejecutar todo*. Los resultados se guardan en Google Drive
(`MyDrive/mopac_gpu_runs/RESULTS_NAME`); si la sesión se corta, vuelva a ejecutar todo y lo terminado se salta. La
última celda descarga un zip.

## 1. Configuración

In [ ]:
COMMIT = 'db2e4ad4'           # qmmm_scan.py
SEEDS = [1, 2, 3]              # una réplica por semilla (equilibrado MM y velocidades propios)
SCAN = 'E:195:OG-I:15:C'       # coordenada 1: Ser195 OG – Lys15 C (BPTI)
TO, STEP = 1.45, 0.1           # Å: desde el reactivo hasta TO
SCAN2 = 'E:195:OG-E:195:HG-E:57:NE2'   # coordenada 2: d(OG–HG) − d(HG–NE2), transferencia del protón
FROM2, TO2, STEP2 = -1.1, 1.1, 0.2
MIN_ITER, MIN_ITER_START = 300, 2000
BACK = True                    # volver al reactivo por el borde de la malla (histéresis)
RELEASE, RELEASE_ITER = True, 3000   # minimizar sin restricciones desde cada punto de la última fila
MM_EQUIL_PS = 50
TOLERANCE = 0.001              # criterio SCF relativo de MOPAC
USE_DRIVE = True
QUICK = False                  # True: una semilla, pasos de 0,2 Å, pocas iteraciones (prueba del cuaderno)
REQUIRE_GPU = None             # 'A100' para exigirla; None: cualquier GPU

## 2. GPU, CPU y carpeta de resultados

In [ ]:
import os, re, subprocess, sys, time, shutil, json
from pathlib import Path

def run(cmd, check=True, env=None, log=None):
    # Run a shell command and stream its output to the cell (and to a log file)
    print('$', cmd, flush=True)
    full = dict(os.environ); full.update(env or {})
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=full)
    fh = open(log, 'w') if log else None
    for line in p.stdout:
        print(line, end='', flush=True)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if check and p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

run('nvidia-smi')
gpu = subprocess.run('nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip()
cpu = subprocess.run("lscpu | grep 'Model name' | sed 's/.*: *//'", shell=True, capture_output=True,
                     text=True).stdout.strip()
NCPU = os.cpu_count()
print(f'GPU: {gpu}\nCPU: {cpu}, {NCPU} núcleos lógicos')
CC = gpu.split(',')[-1].strip().replace('.', '')
if REQUIRE_GPU and REQUIRE_GPU not in gpu:
    raise RuntimeError(f'Esta sesión tiene {gpu}, no una {REQUIRE_GPU}. Cambie el tipo de entorno '
                       f'(o ponga REQUIRE_GPU = None) y vuelva a ejecutar todo.')
tag = re.sub(r'[^a-z0-9]+', '', gpu.split(',')[0].lower().replace('nvidia', ''))[:12]
RESULTS_NAME = f'enzyme_scan_{tag}_{COMMIT[:8]}' + ('_quick' if QUICK else '')
if QUICK:
    SEEDS, STEP, STEP2, MIN_ITER, MIN_ITER_START, MM_EQUIL_PS, RELEASE_ITER = SEEDS[:1], 0.4, 1.1, 30, 200, 5, 100

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/mopac_gpu_runs')
else:
    BASE = Path('/content/mopac_gpu_runs')
OUT = BASE / RESULTS_NAME
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'env.json').write_text(json.dumps({'gpu': gpu, 'cpu': cpu, 'ncpu': NCPU, 'commit': COMMIT, 'seeds': SEEDS,
                                          'scan': SCAN, 'to': TO, 'step': STEP, 'scan2': SCAN2, 'from2': FROM2,
                                          'to2': TO2, 'step2': STEP2, 'min_iter': MIN_ITER,
                                          'min_iter_start': MIN_ITER_START, 'mm_equil_ps': MM_EQUIL_PS,
                                          'tolerance': TOLERANCE, 'back': BACK, 'release': RELEASE,
                                          'release_iter': RELEASE_ITER}, indent=1))
print('resultados en', OUT)

## 3. Compilar MOPAC con la GPU

~10 minutos.

In [ ]:
SRC, BUILD = Path('/content/mopac_gpu'), Path('/content/build')
run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq gfortran ninja-build '
    'libopenblas-dev liblapack-dev > /dev/null')
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
shutil.rmtree(SRC, ignore_errors=True); shutil.rmtree(BUILD, ignore_errors=True)
run(f'git clone -q https://github.com/juvenalyosa/mopac_gpu.git {SRC}')
run(f'git -C {SRC} checkout -q {COMMIT}')
run(f'git -C {SRC} log -1 --format="%h %ad %s" --date=short')
head = subprocess.run(f'git -C {SRC} rev-parse HEAD', shell=True, capture_output=True, text=True).stdout.strip()
if not head.startswith(COMMIT):
    raise SystemExit(f'HEAD {head} no es COMMIT {COMMIT}')
stamp = OUT / 'commit.txt'
if stamp.exists() and not head.startswith(stamp.read_text().strip()[:8]):
    raise SystemExit(f'{OUT} tiene resultados de {stamp.read_text().strip()}; use otro RESULTS_NAME')
stamp.write_text(head + '\n')
t0 = time.time()
run(f'cmake -S {SRC} -B {BUILD} -G Ninja -DGPU=ON -DTESTS=OFF -DCMAKE_BUILD_TYPE=Release -DCUDA_ARCHS={CC} '
    f'-DENABLE_MKL=OFF -DBLA_VENDOR=OpenBLAS > /content/cmake.log 2>&1 || (tail -40 /content/cmake.log; exit 1)')
run(f'cmake --build {BUILD} --target mopac --parallel {NCPU} > /content/build.log 2>&1 '
    f'|| (grep -i -B2 -A6 error /content/build.log | head -80; exit 1)')
LIBMOPAC = next(BUILD.glob('libmopac.so*'))
print(f'compilado en {time.time() - t0:.0f} s: {LIBMOPAC}')

## 4. OpenMM y CuPy

In [ ]:
m = re.search(r'release (\d+)', subprocess.run('nvcc --version', shell=True, capture_output=True, text=True).stdout)
cupy_pkg = f'cupy-cuda{m.group(1) if m else 12}x'
other = [p for p in subprocess.run('pip list 2>/dev/null', shell=True, capture_output=True, text=True).stdout.split()
         if p.startswith('cupy-cuda') and p != cupy_pkg]
if other:
    run('pip uninstall -y -q ' + ' '.join(other), check=False)
run(f"pip install -q 'openmm[cuda12]==8.6.1' {cupy_pkg} 2>&1 | tail -2", check=False)
run(f"{sys.executable} -c \"import openmm as m; print('OpenMM', m.__version__, "
    f"[m.Platform.getPlatform(i).getName() for i in range(m.Platform.getNumPlatforms())])\"", check=False)
run(f"cd {SRC}/scripts/qmmm && {sys.executable} -c \"from qmmm_pme import gpu_available, fused_far_field_available; "
    f"print('QM-PME grid on the GPU (CuPy):', gpu_available(), '; fused kernels:', fused_far_field_available())\"",
    check=False)

## 5. Estructura (PDB 2PTC)

In [ ]:
PDB = Path('/content/2PTC.pdb')
if not PDB.exists():
    run(f'wget -q -O {PDB} https://files.rcsb.org/download/2PTC.pdb')
print(sum(1 for l in PDB.read_text().splitlines() if l.startswith('ATOM')), 'átomos ATOM en 2PTC')

## 6. Mapas

Una carpeta por semilla (`seed_N/`): `scan.csv` (una fila por punto: objetivos y valores alcanzados de las dos
coordenadas, energía sin las restricciones, calor QM, distancias vigiladas, tiempo), `scan_frames.pdb` (el sitio activo
en cada punto) y `scan.log`. Las semillas terminadas se saltan; una semilla interrumpida empieza de nuevo.

In [ ]:
WATCH = 'E:195:OG-E:195:HG,E:195:HG-E:57:NE2,I:15:C-I:15:O,I:15:C-I:16:N,E:57:HD1-E:102:OD2'
for seed in SEEDS:
    d = OUT / f'seed_{seed}'
    if (d / 'scan.csv').exists():
        print(f'semilla {seed}: ya terminada'); continue
    d.mkdir(exist_ok=True)
    shutil.copy2(PDB, d / PDB.name)
    t0 = time.time()
    rc = run(f'cd {d} && {sys.executable} {SRC}/scripts/qmmm/qmmm_scan.py {PDB.name} --libmopac {LIBMOPAC} '
             f'--qm-residues E:57:sc,E:102:sc,E:195:sc,I:15,I:16 --variants E:57=HID --scan {SCAN} --to {TO} '
             f'--step {STEP} --scan2 {SCAN2} --from2 {FROM2} --to2 {TO2} --step2 {STEP2} '
             f'--watch {WATCH} --tolerance {TOLERANCE} --min-iter {MIN_ITER} '
             f'--min-iter-start {MIN_ITER_START} --mm-equil-ps {MM_EQUIL_PS} --seed {seed} --platform CUDA'
             + (' --back' if BACK else '') + (f' --release --release-iter {RELEASE_ITER}' if RELEASE else ''),
             check=False, log=str(d / 'scan.log'))
    print(f'semilla {seed}: exit {rc}, {time.time() - t0:.0f} s', flush=True)

## 7. Figuras

In [ ]:
import csv
import numpy as np
import matplotlib.pyplot as plt

def read(d):
    with open(d / 'scan.csv') as fh:
        rows = list(csv.DictReader(fh))
    for r in rows:
        for k in r:
            if k != 'window':
                r[k] = float(r[k])
    return rows

done = [s for s in SEEDS if (OUT / f'seed_{s}' / 'scan.csv').exists()]
fig, axs = plt.subplots(1, max(1, len(done)), figsize=(5.2 * max(1, len(done)), 4.2), squeeze=False)
for ax, seed in zip(axs[0], done):
    rows = read(OUT / f'seed_{seed}')
    e0 = rows[0]['e']
    g = [r for r in rows if r['window'].startswith('g')]
    x = np.array([r['d'] for r in g]); y = np.array([r['d2'] for r in g]); z = np.array([(r['e'] - e0) / 4.184 for r in g])
    cs = ax.tricontourf(x, y, z, levels=20, cmap='viridis')
    ax.plot(x, y, '.', color='w', ms=2)
    ax.plot(rows[0]['d'], rows[0]['d2'], '*', color='r', ms=12, label='reactivo')
    fig.colorbar(cs, ax=ax, label='energía − reactivo (kcal/mol)')
    ax.invert_xaxis()
    ax.set_xlabel('Ser195 OG – Lys15 C (Å)'); ax.set_ylabel('d(OG–HG) − d(HG–NE2) (Å)')
    ax.set_title(f'semilla {seed}'); ax.legend(loc='upper right')
    end = [r for r in g if abs(r['d'] - min(x)) < 0.06]
    best = min(end, key=lambda r: r['e'])
    print(f"semilla {seed}: reactivo OG–C {rows[0]['d']:.2f} Å, coordenada del protón {rows[0]['d2']:+.2f} Å; "
          f"en OG–C {best['d']:.2f} Å la energía mínima es {(best['e'] - e0) / 4.184:+.1f} kcal/mol, con el protón en "
          f"{best['d2']:+.2f} Å (HG···NE2 {best['E:195:HG-E:57:NE2']:.2f} Å); puntos: {len(g)}")
    # ida y vuelta en los mismos objetivos
    ida = {(round(r['target'], 3), round(r['target2'], 3)): r for r in g}
    dif = [(r['e'] - ida[k]['e']) / 4.184 for r in rows if r['window'].startswith('b')
           for k in [(round(r['target'], 3), round(r['target2'], 3))] if k in ida]
    if dif:
        print(f"   vuelta − ida en {len(dif)} puntos: de {min(dif):+.1f} a {max(dif):+.1f} kcal/mol; "
              f"en el punto de partida {dif[-1]:+.1f} kcal/mol")
    # sin restricciones: de dónde sale y adónde llega cada minimización (flechas en el mapa)
    byname = {r['window']: r for r in rows}
    for r in rows:
        if r['window'].startswith('r_'):
            a0 = byname[r['window'][2:]]
            ax.annotate('', xy=(r['d'], r['d2']), xytext=(a0['d'], a0['d2']),
                        arrowprops=dict(arrowstyle='->', color='r', lw=1))
            print(f"   sin restricciones desde {a0['window']} (OG–C {a0['d']:.2f} Å, protón {a0['d2']:+.2f} Å, "
                  f"{(a0['e'] - e0) / 4.184:+.1f} kcal/mol) -> OG–C {r['d']:.2f} Å, protón {r['d2']:+.2f} Å, "
                  f"{(r['e'] - e0) / 4.184:+.1f} kcal/mol (OG–HG {r['E:195:OG-E:195:HG']:.2f}, "
                  f"HG···NE2 {r['E:195:HG-E:57:NE2']:.2f} Å)")
fig.tight_layout()
fig.savefig(OUT / 'enzyme_scan.png', dpi=150)
plt.show()

## 8. Descargar

In [ ]:
tmp = Path('/content/_enzyme') / RESULTS_NAME
shutil.rmtree(tmp.parent, ignore_errors=True)
tmp.mkdir(parents=True)
for f in ('env.json', 'commit.txt', 'enzyme_scan.png'):
    if (OUT / f).exists():
        shutil.copy2(OUT / f, tmp / f)
for d in sorted(OUT.glob('seed_*')):
    (tmp / d.name).mkdir()
    for f in ('scan.csv', 'scan_frames.pdb', 'scan.log'):
        if (d / f).exists():
            shutil.copy2(d / f, tmp / d.name / f)
z = shutil.make_archive(f'/content/{RESULTS_NAME}', 'zip', tmp.parent)
print(z, f'{os.path.getsize(z) / 1e6:.1f} MB')
from google.colab import files
files.download(z)